# Order Payments table: cleaning

Run from the folder that contains raw/ and cleaned/ (your VS Code workspace root).
Needs cleaned/orders_clean.csv and cleaned/order_items_clean.csv from the earlier notebooks.

Decisions and findings for this table are written up in `cleaning_notes/README_order_payments.md`.

### 1. Imports and settings

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

RAW = Path('raw')
CLEAN = Path('cleaned')
CLEAN.mkdir(exist_ok=True)

### 2. Load the raw payments file

In [2]:
order_payments = pd.read_csv(RAW / 'olist_order_payments_dataset.csv')

### 3. First inspection: structure, nulls, preview, payment methods

In [3]:
order_payments.info()
print(order_payments.isna().sum())                  # expect all zeros
print(order_payments.head())
print(order_payments['payment_type'].value_counts())
# credit_card 76,795 / boleto 19,784 / voucher 5,775 / debit_card 1,529 / not_defined 3

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64
                           order_id  payment_sequential payment_type  \
0  b81ef226f3fe1789b1e8b2acac839d17                   1  credit_card   
1  a9810da82917af2d9aefd1278f1dcfa0                   1  credit_card   
2  25e8ea4e93396b6fa0d3dd708e76c1bd                   1  credit_card   
3  ba78997921bbcdc1373bb41e913ab953             

### 4. Key check: one row per payment, so the key is the PAIR (order_id, payment_sequential)

In [4]:
print(order_payments.duplicated(subset=['order_id', 'payment_sequential']).sum())   # expect 0
print(order_payments['order_id'].nunique())                  # 99,440 orders have payments
print(order_payments['payment_installments'].max())          # 24

# (order_id, payment_type) repeats 2,200 times. That is NOT a duplicate problem:
# an order can legitimately be paid twice with the same method (two vouchers, split card payment).
print(order_payments.duplicated(subset=['order_id', 'payment_type']).sum())

0
99440
24
2200


### 5. Numeric profile and the odd rows

In [5]:
print(order_payments[['payment_installments', 'payment_value']].describe())

print(order_payments[order_payments['payment_value'] <= 0])            # 9 rows: 6 voucher + 3 not_defined
print(order_payments[order_payments['payment_installments'] < 1])      # 2 credit_card rows with 0 installments
print(order_payments[order_payments['payment_type'] == 'not_defined']) # 3 rows, all value 0.0

       payment_installments  payment_value
count         103886.000000  103886.000000
mean               2.853349     154.100380
std                2.687051     217.494064
min                0.000000       0.000000
25%                1.000000      56.790000
50%                1.000000     100.000000
75%                4.000000     171.837500
max               24.000000   13664.080000
                                order_id  payment_sequential payment_type  \
19922   8bcbe01d44d147f901cd3192671144db                   4      voucher   
36822   fa65dad1b0e818e3ccc5cb0e39231352                  14      voucher   
43744   6ccb433e00daae1283ccc956189c82ae                   4      voucher   
51280   4637ca194b6387e2d538dc89b124b0ee                   1  not_defined   
57411   00b1cb0320190ca0daa2c88b35206009                   1  not_defined   
62674   45ed6e85398a87c253db47c2d9f48216                   3      voucher   
77885   fa65dad1b0e818e3ccc5cb0e39231352                  13      voucher 

### 6. Join integrity with orders

In [6]:
orders = pd.read_csv(CLEAN / 'orders_clean.csv', usecols=['order_id', 'order_status'])

print(order_payments['order_id'].isin(orders['order_id']).value_counts())   # True 103,886 (no orphan payments)
no_pay = ~orders['order_id'].isin(order_payments['order_id'])
print(orders.loc[no_pay])                    # 1 order, status delivered (a source-data gap)

nd = order_payments.loc[order_payments['payment_type'] == 'not_defined', 'order_id']
print(orders[orders['order_id'].isin(nd)])   # all 3 are canceled

order_id
True    103886
Name: count, dtype: int64
                               order_id order_status
30710  bfbd0f9bdef84302105ad712db648a6c    delivered
                               order_id order_status
1130   00b1cb0320190ca0daa2c88b35206009     canceled
39919  4637ca194b6387e2d538dc89b124b0ee     canceled
40235  c8c528189310eaa44a745b8d9d26908b     canceled


### 7. Flag problem rows (kept, not dropped)

In [7]:
order_payments['has_data_issue'] = (
    (order_payments['payment_installments'] < 1) | (order_payments['payment_value'] == 0)
)
print(order_payments['has_data_issue'].value_counts())      # False 103,875 / True 11

has_data_issue
False    103875
True         11
Name: count, dtype: int64


### 8. Reconcile payments against item values, per order

In [8]:
items = pd.read_csv(CLEAN / 'order_items_clean.csv', usecols=['order_id', 'item_total'])

items_total = items.groupby('order_id')['item_total'].sum().rename('items_total')
paid_total = order_payments.groupby('order_id')['payment_value'].sum().rename('paid_total')

rec = pd.concat([items_total, paid_total], axis=1)
print(rec.isna().sum())          # items_total 775 (orders without items) / paid_total 1 (the unpaid order)

both = rec.dropna().copy()
both['diff'] = (both['paid_total'] - both['items_total']).round(2)
print(len(both))                              # 98,665
print((both['diff'].abs() > 0.01).sum())      # 303 mismatches (0.3%)
print(both['diff'].describe())                # min -51.62 / max 182.81 / median 0

print(rec.loc[rec['items_total'].isna(), 'paid_total'].sum().round(2))   # 162,591.95 paid on orders with no items

# Payment types of the 303 mismatched orders
pt = order_payments.groupby('order_id')['payment_type'].agg(lambda s: '+'.join(sorted(s.unique())))
mism = both[both['diff'].abs() > 0.01]
print(mism.join(pt)['payment_type'].value_counts())
# credit_card 275 / boleto 14 / credit_card+voucher 7 / debit_card 7

items_total    775
paid_total       1
dtype: int64
98665
303
count    98665.000000
mean         0.029092
std          1.129221
min        -51.620000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: diff, dtype: float64
162591.95
payment_type
credit_card            275
boleto                  14
credit_card+voucher      7
debit_card               7
Name: count, dtype: int64


### 9. Save the cleaned table and verify the file on disk

In [9]:
order_payments.to_csv(CLEAN / 'order_payments_clean.csv', index=False)

check = pd.read_csv(CLEAN / 'order_payments_clean.csv')
print(check.shape)                                                          # expect (103886, 6)
print(check.duplicated(subset=['order_id', 'payment_sequential']).sum())    # expect 0
print(check['has_data_issue'].value_counts())                               # True 11 / False 103,875
print(check['payment_value'].sum().round(2))                                # 16,008,872.12

(103886, 6)
0
has_data_issue
False    103875
True         11
Name: count, dtype: int64
16008872.12
